In [ ]:
from os import listdir as ls
import pandas as pd
import seaborn as sns
import pycountry
from matplotlib.pyplot import subplots
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.utils import get_countries_by_continent

set_matplotlib_formats("svg")

https://unstats.un.org/unsd/methodology/m49/overview/#

In [ ]:
job_path = OUTPUTS_PATH / "46622452"
countries = ls(job_path)
analyses = [a for a in ANALYSIS_NAMES if "no_mob" not in a]

In [ ]:
def get_country_region_map():
    filename = DATA_PATH / "mapping/UNSD — Methodology.csv"
    return pd.read_csv(filename, sep=";", index_col="ISO-alpha3 Code")["Sub-region Name"]

In [ ]:
idatas, _ = get_idatas_for_mob_type(job_path, countries, "g_mob")
mob_exp_df = pd.DataFrame(columns=idatas)
for iso3 in idatas:
    mob_exp_df[iso3] = idatas[iso3].posterior["mob_exp"].to_dataframe()

In [ ]:
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)

In [ ]:
oceania = ["AUS", "FJI", "NZL", "SGP"]
north_america = ["USA", "CAN", "MEX", "GTM", "BLZ", "SLV", "HND", "NIC", "CRI", "PAN", "CUB", "HTI", "DOM", "JAM", "TTO", "BRB", "BHS", "GRD", "LCA", "VCT", "ATG", "KNA", "PRI"]
south_america = ["ARG", "BOL", "BRA", "CHL", "COL", "ECU", "GUY", "PRY", "PER", "SUR", "URY", "VEN", "ABW"]
west_africa = ["BEN", "BFA", "CPV", "CIV", "GMB", "GHA", "GIN", "GNB", "LBR", "MLI", "NER", "NGA", "SEN", "SLE", "TGO"]
east_africa = ["BDI", "COM", "DJI", "ERI", "ETH", "KEN", "MDG", "MWI", "MOZ", "RWA", "SOM", "SSD", "TZA", "UGA", "ZMB"]
southern_africa = ["AGO", "BWA", "LSO", "NAM", "SWZ", "ZAF", "ZMB", "ZWE"]
central_africa = ["CMR", "CAF", "TCD", "COG", "COD", "GNQ", "GAB", "STP"]

grouping = {}
for c in all_countries:
    if c in oceania:
        region = "Oceania"
    elif c in north_america:
        region = "North America"
    elif c in south_america:
        region = "South America"
    elif c in west_africa:
        region = "West Africa"
    elif c in southern_africa:
        region = "Southern Africa"
    elif c in central_africa or c in east_africa:
        region = "Central and East Africa"
    else:
        region = get_country_region_map()[c]
    if region in ["South-eastern Asia", "Eastern Asia"]:
        region = "Eastern and South-eastern Asia"
    if region not in grouping:
        grouping[region] = [c]
    else:
        grouping[region].append(c)

In [ ]:
def get_country_short_name(iso3):
    info = pycountry.countries.lookup(iso3)
    abbrevs = {
        "GBR": "UK",
        "ARE": "UAE",
        "RUS": "Russia",
        "DOM": "Domin. Rep.",
        "BIH": "Bosnia Herz",
        "AFG": "Afghan.",
    }
    if hasattr(info, "common_name"):
        return info.common_name
    elif iso3 in abbrevs:
        return abbrevs[iso3]
    else:
        return info.name

In [ ]:
fig, axes = subplots(3, 5, figsize=[12, 12], sharey=True)
flat_axes = axes.ravel()
avail_countries = mob_exp_df.columns
for c, (region, countries) in enumerate(grouping.items()):
    ax = flat_axes[c]
    ax.set_title(region)
    cs = [c for c in countries if c in avail_countries]
    df = mob_exp_df[cs].copy()
    df.columns = df.columns.map(get_country_short_name)
    sns.violinplot(df, ax=ax)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=90)
fig.tight_layout()